In [ ]:
from pathlib import Path


import torch
import pandas as pd
from tqdm.notebook import tqdm

from rtnls_inference import (
    RegressionEnsemble,
)

In [ ]:
ds_path = Path("../samples/fundus")

# input folders. these are the folders where we stored the preprocessed images
rgb_path = ds_path / "rgb"
device = torch.device("cuda:0")  # device to use for inference

In [ ]:
rgb_paths = sorted(list(rgb_path.glob("*.png")))

In [ ]:
rgb_paths

In [ ]:
ensemble = RegressionEnsemble.from_release("odfd_march25.pt").to(device)

dataloader = ensemble._make_inference_dataloader(
    rgb_paths,
    num_workers=8,
    preprocess=False,
    batch_size=8,
)

In [ ]:
output_ids, outputs = [], []
with torch.no_grad():
    for batch in tqdm(dataloader):
        if len(batch) == 0:
            continue

        im = batch["image"].to(device)
        val = ensemble.forward(im).mean(dim=0) # average the model dimension

        output_ids += batch["id"]
        outputs.append(val)

In [ ]:
df = pd.DataFrame(torch.cat(outputs), index=output_ids)

In [ ]:
df